In [ ]:
# ==================== 第一部分：环境准备与数据加载 ====================

import os
# 防显存碎片化 OOM：让 CUDA 分配器使用可扩展段（须在 import torch 之前设置）
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import transforms, datasets, models
from torchvision.models import ResNet18_Weights

# 评估指标库
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             confusion_matrix, classification_report)

# 固定随机种子
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 数据集根目录：Kaggle 上优先用挂载的 input 数据集（只读 .tar.gz），解压一次到可写目录；
# 否则回退本地 ./data（已解压则直接读，都没有才联网下载那个慢源）。
import glob, tarfile

def _prepare_cifar_root():
    # 1) 本地工作区已有解压好的 cifar-10-batches-py 就直接用
    for root in ('./data', '/kaggle/working/data'):
        if os.path.isdir(os.path.join(root, 'cifar-10-batches-py')):
            return root
    # 2) Kaggle input 里已有解压好的 cifar-10-batches-py，直接只读读取（最快，免解压）
    dirs = glob.glob('/kaggle/input/**/cifar-10-batches-py', recursive=True)
    if dirs:
        root = os.path.dirname(dirs[0])
        print(f"✓ 直接读取 Kaggle input 已解压数据集: {root}")
        return root
    # 3) input 里只有 tar 包，解压到可写目录（纯本地，几秒，不联网）
    hits = glob.glob('/kaggle/input/**/cifar-10-python.tar.gz', recursive=True)
    if hits:
        root = '/kaggle/working/data'
        os.makedirs(root, exist_ok=True)
        print(f"📦 从 Kaggle input 解压数据集: {hits[0]}")
        with tarfile.open(hits[0]) as t:
            t.extractall(root)          # 生成 {root}/cifar-10-batches-py
        return root
    # 4) 都没有：用 ./data，后续 download=True 会联网下载
    return './data'

DATA_ROOT = _prepare_cifar_root()

def cifar10_need_download(root=DATA_ROOT):
    """torchvision 解压后会生成 cifar-10-batches-py 目录；存在即视为本地已就绪，无需联网。"""
    ready = os.path.isdir(os.path.join(root, 'cifar-10-batches-py'))
    if ready:
        print(f"✓ 检测到本地数据集，直接加载: {os.path.abspath(root)}")
    else:
        print(f"⬇ 未检测到本地数据集，将下载到: {os.path.abspath(root)}")
    return not ready

# -------------------- 数据预处理 --------------------
# 训练集：原生 32×32 + CIFAR 标准数据增强
train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 验证集/测试集：仅 ToTensor + Normalize
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 加载 CIFAR-10 数据集（本地优先；本地已存在则不会联网）
_need_download = cifar10_need_download(DATA_ROOT)
full_train_dataset = datasets.CIFAR10(
    root=DATA_ROOT, train=True, download=_need_download, transform=train_transform
)
# 验证集用同一批训练数据但套 test_transform（无增广）：单独再建一个实例，
# 避免与训练集共享底层 dataset —— 否则改其一的 transform 会连带污染另一个。
val_base_dataset = datasets.CIFAR10(
    root=DATA_ROOT, train=True, download=False, transform=test_transform
)
test_dataset = datasets.CIFAR10(
    root=DATA_ROOT, train=False, download=_need_download, transform=test_transform
)

# 按同一套随机索引做 9:1 划分：训练子集取自增广实例，验证子集取自无增广实例
_split_gen = torch.Generator().manual_seed(42)
_perm = torch.randperm(len(full_train_dataset), generator=_split_gen).tolist()
_split = int(0.9 * len(full_train_dataset))
train_dataset = Subset(full_train_dataset, _perm[:_split])
val_dataset = Subset(val_base_dataset, _perm[_split:])

# 创建 DataLoader
BATCH_SIZE = 128
NUM_WORKERS = 2

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

# 类别名称
CLASS_NAMES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']
NUM_CLASSES = len(CLASS_NAMES)

print(f"训练集样本数: {len(train_dataset)}")
print(f"验证集样本数: {len(val_dataset)}")
print(f"测试集样本数: {len(test_dataset)}")
print(f"类别总数: {NUM_CLASSES}")

In [ ]:
# ==================== 第二部分：数据集样本可视化 ====================

# 使用未归一化的原始图像显示（避免颜色异常）
raw_transform = transforms.Compose([
    transforms.ToTensor()
])
# 本地优先：复用第一部分的 DATA_ROOT，本地已存在则不会联网
raw_dataset = datasets.CIFAR10(root=DATA_ROOT, train=True,
                               download=cifar10_need_download(DATA_ROOT),
                               transform=raw_transform)

# 获取每个类别的第一张图
class_to_img = {}
for img, label in raw_dataset:
    if label not in class_to_img:
        class_to_img[label] = img
    if len(class_to_img) == NUM_CLASSES:
        break

# 绘制 2x5 网格
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
axes = axes.flatten()
for i in range(NUM_CLASSES):
    img_tensor = class_to_img[i]
    img_np = img_tensor.permute(1, 2, 0).numpy()
    axes[i].imshow(img_np)
    axes[i].set_title(CLASS_NAMES[i], fontsize=12)
    axes[i].axis('off')
plt.suptitle('CIFAR-10 各类别样本展示 (原生 32×32)', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# ==================== 第三部分：ResNet-18 模型构建 ====================

def create_resnet18(num_classes=10, pretrained=False):
    # CIFAR 版 ResNet-18：原生 32×32 输入
    if pretrained:
        model = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    else:
        model = models.resnet18(weights=None)
    # 把 ImageNet 的 7x7 stride2 + maxpool 换成 3x3 stride1 + 去 maxpool，
    # 否则 32x32 一上来就被压到 8x8，丢失太多空间信息。
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model

model = create_resnet18(num_classes=NUM_CLASSES, pretrained=False).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"总参数量: {total_params:,}")
print(f"可训练参数量: {trainable_params:,}")

In [ ]:
# ==================== 第四部分：早停机制与训练辅助函数 ====================

class EarlyStopping:
    """早停机制：当验证指标在 patience 轮内未提升时停止训练"""
    def __init__(self, patience=5, delta=0.001, mode='max', verbose=True):
        """
        Args:
            patience (int): 容忍多少个 epoch 无提升
            delta (float): 提升的最小阈值
            mode (str): 'max'（监控指标越大越好，如准确率）或 'min'（监控指标越小越好，如损失）
            verbose (bool): 是否打印早停信息
        """
        self.patience = patience
        self.delta = delta
        self.mode = mode
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        if mode == 'max':
            self.best_score = -float('inf')
        else:
            self.best_score = float('inf')

    def __call__(self, current_score):
        if self.mode == 'max':
            score_improved = current_score > self.best_score + self.delta
        else:
            score_improved = current_score < self.best_score - self.delta

        if score_improved:
            self.best_score = current_score
            self.counter = 0
        else:
            self.counter += 1
            if self.verbose:
                print(f"早停计数器: {self.counter}/{self.patience} (当前最佳: {self.best_score:.4f})")
            if self.counter >= self.patience:
                self.early_stop = True

def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    pbar = tqdm(dataloader, desc="Training", leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        pbar.set_postfix({'loss': loss.item(), 'acc': correct/total})
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc

def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        pbar = tqdm(dataloader, desc="Evaluating", leave=False)
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            pbar.set_postfix({'loss': loss.item(), 'acc': correct/total})
    epoch_loss = running_loss / total
    epoch_acc = correct / total
    return epoch_loss, epoch_acc, all_preds, all_labels

In [ ]:
# ==================== 第五部分：模型训练（集成早停） ====================

NUM_EPOCHS = 30               # 设置一个较大的值，早停会自动截断
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4
PATIENCE = 5                  # 连续5个epoch验证准确率不提升则停止

criterion = nn.CrossEntropyLoss()

# CPU 上完整训练极慢；若已有训练好的权重，设 SKIP_TRAINING=True 可直接加载并跳过训练。
SKIP_TRAINING = True  # 已有训练好的干净模型 → 跳过训练，直接加载
CKPT_PATH = '/kaggle/input/notebooks/liangliguo/cifar/resnet18_cifar10_best.pth'  # 干净模型权重路径

if SKIP_TRAINING and os.path.exists(CKPT_PATH):
    print(f"⏭️  跳过训练，直接加载已有模型: {CKPT_PATH}")
    checkpoint = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    best_model_state = checkpoint['model_state_dict']
    history = checkpoint.get('history',
                             {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []})
    best_val_acc = checkpoint.get('best_val_acc', 0.0)
    print(f"✓ 模型加载完成，记录的最佳验证准确率: {best_val_acc:.4f}")
else:
    # 多 GPU：DataParallel 把每个 batch 切分到所有可见 GPU（如 T4×2）。
    # 只包装训练前向用的 train_model；model 仍是原始模块，因此 optimizer 用
    # model.parameters()、保存用 model.state_dict() 都不带 "module." 前缀，
    # 与 checkpoint 及后续单元（cell 11 加载到普通 model）兼容。
    if torch.cuda.device_count() > 1:
        train_model = nn.DataParallel(model)
        print(f"✓ 使用 {torch.cuda.device_count()} 张 GPU（DataParallel）训练")
    else:
        train_model = model

    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

    # 初始化早停（监控验证准确率，越大越好）
    early_stopping = EarlyStopping(patience=PATIENCE, delta=0.001, mode='max', verbose=True)

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0.0
    best_model_state = None

    print("开始训练（启用早停机制）...")
    start_time = time.time()

    for epoch in range(1, NUM_EPOCHS + 1):
        print(f"\n{'='*50}\nEpoch {epoch}/{NUM_EPOCHS}\n{'='*50}")
        train_loss, train_acc = train_epoch(train_model, train_loader, criterion, optimizer, device)
        val_loss, val_acc, _, _ = evaluate(train_model, val_loader, criterion, device)
        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
        print(f"Learning Rate: {current_lr:.6f}")

        # 保存最佳模型
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            print(f"✓ 新的最佳模型！验证准确率: {val_acc:.4f}")

        # 早停检查
        early_stopping(val_acc)
        if early_stopping.early_stop:
            print(f"\n⏹️ 早停触发！验证准确率连续 {PATIENCE} 个 epoch 未提升。")
            break

    training_time = time.time() - start_time
    print(f"\n训练完成！总耗时: {training_time/60:.2f} 分钟")
    print(f"最佳验证准确率: {best_val_acc:.4f}")

    # 加载最佳模型用于测试
    model.load_state_dict(best_model_state)

In [ ]:
# ==================== 第六部分：训练曲线绘制 ====================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Curves'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history['train_acc'], label='Train Acc', marker='o')
axes[1].plot(history['val_acc'], label='Val Acc', marker='s')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy Curves'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ==================== 第七部分：测试集详尽评测 ====================

print("\n" + "="*50)
print("测试集评估与指标计算")
print("="*50)

# 在测试集上评估
test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, criterion, device)

# 计算加权与宏平均指标
precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(test_labels, test_preds, average='weighted')
precision_m, recall_m, f1_m, _ = precision_recall_fscore_support(test_labels, test_preds, average='macro')

# 计算 Top-3 准确率
def top_k_accuracy(outputs, labels, k=3):
    _, pred_topk = outputs.topk(k, 1, True, True)
    pred_topk = pred_topk.t()
    correct = pred_topk.eq(labels.view(1, -1).expand_as(pred_topk))
    return correct[:k].reshape(-1).float().sum(0, keepdim=True) / labels.size(0)

model.eval()
all_outputs = []
with torch.no_grad():
    for images, _ in test_loader:
        images = images.to(device)
        outputs = model(images)
        all_outputs.append(outputs.cpu())
all_outputs = torch.cat(all_outputs, dim=0)
top3_acc = top_k_accuracy(all_outputs, torch.tensor(test_labels), k=3).item()

print("\n📊 整体指标汇总")
print("-" * 40)
print(f"测试准确率 (Top-1)   : {test_acc:.4f}")
print(f"Top-3 准确率         : {top3_acc:.4f}")
print(f"测试损失             : {test_loss:.4f}")
print(f"加权精确率 (Weighted): {precision_w:.4f}")
print(f"加权召回率 (Weighted): {recall_w:.4f}")
print(f"加权 F1 分数         : {f1_w:.4f}")
print(f"宏平均精确率 (Macro) : {precision_m:.4f}")
print(f"宏平均召回率 (Macro) : {recall_m:.4f}")
print(f"宏平均 F1 分数       : {f1_m:.4f}")

# 各类别详细报告
print("\n📋 各类别分类报告")
print("-" * 40)
print(classification_report(test_labels, test_preds, target_names=CLASS_NAMES, digits=4))

# 混淆矩阵可视化
cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix - CIFAR-10 Test Set', fontsize=14)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.xticks(rotation=45); plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ==================== 第八部分：关键指标对比柱状图 ====================

metrics_names = ['Top-1 Acc', 'Top-3 Acc', 'Weighted F1', 'Macro F1']
metrics_values = [test_acc, top3_acc, f1_w, f1_m]

plt.figure(figsize=(10, 6))
bars = plt.bar(metrics_names, metrics_values, color=['#2ecc71', '#3498db', '#9b59b6', '#e67e22'])
plt.ylim(0, 1.0)
plt.ylabel('Score')
plt.title('ResNet-18 Performance on CIFAR-10 Test Set')
for bar, val in zip(bars, metrics_values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.4f}', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ==================== 第九部分：保存模型权重 ====================

torch.save({
    'model_state_dict': best_model_state,
    'num_classes': NUM_CLASSES,
    'class_names': CLASS_NAMES,
    'history': history,
    'best_val_acc': best_val_acc,
    'test_acc': test_acc,
    'test_f1': f1_w
}, 'resnet18_cifar10_best.pth')

print("模型已保存为 'resnet18_cifar10_best.pth'")

In [ ]:
# ==================== 生成模型下载链接 ====================
import os
from IPython.display import FileLink, display

# 假设已保存模型文件
model_path = 'resnet18_cifar10_best.pth'

if os.path.exists(model_path):
    print(f"✅ 模型文件已就绪，大小: {os.path.getsize(model_path) / 1024**2:.2f} MB")
    display(FileLink(model_path, result_html_prefix="点击此处下载模型: "))
else:
    print("❌ 未找到模型文件，请先保存。")

In [ ]:
# !pip install torchattacks

In [ ]:
# ==================== 优化版本（解决显存不足与速度慢） ====================
# 关键修正：
#   1) 攻击全部在 [0,1] 像素空间进行，归一化放进模型（Normalize 包装层），
#      避免 torchattacks 的 clamp(0,1) 把归一化图像截毁。
#   2) CustomFGSM 与 torchattacks 攻击口径一致，FGSM 同样 clamp 到 [0,1]。
#   3) 修正提前停止条件：按“样本数”而非“batch 数”判断。
import os
import math
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from matplotlib.lines import Line2D
from tqdm import tqdm
from torch.utils.data import DataLoader, Subset
from torchvision import transforms, datasets
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import gc

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# CIFAR-10 类别名（攻击单元自包含，避免依赖训练部分的 CLASS_NAMES）
CIFAR10_CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                   'dog', 'frog', 'horse', 'ship', 'truck']

# ---------- 全局绘图风格（统一所有攻击图的字体 / 网格 / 留白 / DPI）----------
plt.rcParams.update({
    'figure.dpi': 120, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'font.size': 11, 'axes.titlesize': 13, 'axes.titleweight': 'bold',
    'axes.labelsize': 11, 'legend.fontsize': 10,
    'axes.grid': True, 'grid.linestyle': '--', 'grid.alpha': 0.3, 'axes.axisbelow': True,
    'axes.spines.top': False, 'axes.spines.right': False,
    'legend.frameon': True, 'legend.framealpha': 0.9,
})
# 各组（干净 / 各攻击）统一配色（seaborn 'deep' 调色板）
ATTACK_COLORS = {'Clean': '#4C72B0', 'FGSM': '#DD8452',
                 'DeepFool': '#55A868', 'APGD': '#C44E52'}
# 各「类别」固定配色：类别索引 c → tab10 第 c 色，所有 t-SNE 图共用，保证跨图一致
CLASS_COLORS = list(plt.get_cmap('tab10').colors)

# ---------- 模型定义 ----------
def create_resnet18(num_classes=10, pretrained=False):
    from torchvision.models import resnet18, ResNet18_Weights
    if pretrained:
        model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
    else:
        model = resnet18(weights=None)
    # CIFAR 版 stem：与基础训练单元保持一致（原生 32x32）
    model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)
    return model

model = create_resnet18(num_classes=10, pretrained=False)
CKPT_PATH = '/kaggle/input/notebooks/liangliguo/cifar/resnet18_cifar10_best.pth'  # 干净（基础训练）模型权重
checkpoint = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()
print("Model loaded successfully")

# ---------- 归一化包装层：把归一化放进模型，攻击在 [0,1] 空间进行 ----------
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD  = [0.229, 0.224, 0.225]

class Normalize(nn.Module):
    def __init__(self, mean, std):
        super().__init__()
        self.register_buffer('mean', torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer('std',  torch.tensor(std).view(1, 3, 1, 1))

    def forward(self, x):
        return (x - self.mean) / self.std

# atk_model 接收 [0,1] 图像，内部先归一化再过 ResNet；
# 训练时网络见到的就是归一化分布，因此干净准确率保持一致。
atk_model = nn.Sequential(Normalize(NORM_MEAN, NORM_STD), model).to(device).eval()
inner_model = model  # 特征 hook 仍挂在内部 ResNet 上

# ---------- 数据加载（不做 Normalize，保持 [0,1]）----------
test_transform = transforms.Compose([
    transforms.ToTensor(),  # → [0,1]（原生 32x32），归一化交给 atk_model 内部完成
])
# 复用第一部分自动探测的 DATA_ROOT（Kaggle input / 本地 / 解压）；找不到才回退 ./data
DATA_ROOT = _prepare_cifar_root() if '_prepare_cifar_root' in globals() else './data'
_need_download = not os.path.isdir(os.path.join(DATA_ROOT, 'cifar-10-batches-py'))
print(f"{'⬇ 下载数据集到' if _need_download else '✓ 使用本地数据集'}: {os.path.abspath(DATA_ROOT)}")
test_dataset = datasets.CIFAR10(root=DATA_ROOT, train=False,
                                download=_need_download, transform=test_transform)

# 评估池：用全部测试集 10000 张（攻击各自按 max_samples / num_batches 取所需）
subset_indices = list(range(10000))
test_subset = Subset(test_dataset, subset_indices)
test_loader = DataLoader(test_subset, batch_size=32, shuffle=False, num_workers=2)  # 减小batch size

# ---------- 自定义 FGSM 攻击（在 [0,1] 空间，结尾 clamp 到 [0,1]）----------
class CustomFGSM:
    def __init__(self, model, eps=8 / 255):
        self.model = model      # 传入 atk_model（内部含归一化）
        self.eps = eps          # eps 为 [0,1] 像素空间预算

    def __call__(self, images, labels):
        images = images.clone().detach().requires_grad_(True)
        outputs = self.model(images)
        loss = torch.nn.functional.cross_entropy(outputs, labels)
        grad = torch.autograd.grad(loss, images, retain_graph=False, create_graph=False)[0]
        adv_images = images + self.eps * grad.sign()
        adv_images = torch.clamp(adv_images, 0, 1)   # 与 torchattacks 口径一致
        return adv_images.detach()

# ---------- 特征提取（干净样本，前向走 atk_model，hook 挂在内部 ResNet）----------
def extract_features(fwd_model, dataloader, device, max_samples=300):
    fwd_model.eval()
    features = []
    labels = []
    activation = {}
    def hook_fn(module, input, output):
        activation['feat'] = output.detach()
    handle = inner_model.avgpool.register_forward_hook(hook_fn)
    collected = 0
    # 进度条总长按实际要跑的 batch 数计算，凑够 max_samples 即停，进度条到 100%
    total_batches = math.ceil(max_samples / dataloader.batch_size)
    with torch.no_grad():
        for imgs, lbls in tqdm(dataloader, total=total_batches, desc=f"Extracting features (n={max_samples})"):
            if collected >= max_samples:
                break
            imgs = imgs.to(device)
            _ = fwd_model(imgs)
            feat = activation['feat'].squeeze(-1).squeeze(-1).cpu()
            features.append(feat)
            labels.append(lbls[:feat.size(0)])
            collected += feat.size(0)
            del imgs, _
    handle.remove()
    features = torch.cat(features, dim=0)[:max_samples].numpy()
    labels = torch.cat(labels, dim=0)[:max_samples].numpy()
    return features, labels

# ---------- 生成对抗样本（优化版，显存友好） ----------
def generate_adversarial_optimized(attack, name, fwd_model, dataloader, device, max_samples=1000):
    """
    优化版：只生成对抗样本和扰动，不同时提取特征（特征单独提取）
    返回：adv_images, orig_images, labels, l2_norms
    """
    fwd_model.eval()
    adv_images_list = []
    orig_images_list = []
    labels_list = []
    l2_norms = []
    collected = 0

    # 进度条总长按实际要跑的 batch 数计算（凑够 max_samples 即停）；生成默认 1000 张供直方图 / t-SNE 用
    total_batches = math.ceil(max_samples / dataloader.batch_size)
    for imgs, lbls in tqdm(dataloader, total=total_batches, desc=f"Generating {name} (n={max_samples})"):
        if collected >= max_samples:
            break
        imgs, lbls = imgs.to(device), lbls.to(device)
        # 生成对抗样本（detach：部分 torchattacks 攻击如 APGD 返回的张量仍带 grad）
        adv = attack(imgs, lbls).detach()
        # 计算 L2 扰动
        with torch.no_grad():
            l2 = torch.norm((adv - imgs).view(imgs.size(0), -1), dim=1).cpu()
        adv_images_list.append(adv.cpu())
        orig_images_list.append(imgs.detach().cpu())
        labels_list.append(lbls.cpu())
        l2_norms.append(l2)
        collected += imgs.size(0)
        del imgs, lbls, adv

    adv_imgs = torch.cat(adv_images_list, dim=0)[:max_samples]
    orig_imgs = torch.cat(orig_images_list, dim=0)[:max_samples]
    labels = torch.cat(labels_list, dim=0)[:max_samples]
    l2_norms = torch.cat(l2_norms, dim=0)[:max_samples].numpy()
    return adv_imgs, orig_imgs, labels, l2_norms

# ---------- 从对抗样本中提取特征 ----------
def extract_features_from_adv(fwd_model, adv_imgs, batch_size=32):
    """分批提取对抗样本的特征，避免显存爆炸"""
    fwd_model.eval()
    features = []
    activation = {}
    def hook_fn(module, input, output):
        activation['feat'] = output.detach()
    handle = inner_model.avgpool.register_forward_hook(hook_fn)

    with torch.no_grad():
        for i in range(0, len(adv_imgs), batch_size):
            batch = adv_imgs[i:i+batch_size].to(device)
            _ = fwd_model(batch)
            feat = activation['feat'].squeeze(-1).squeeze(-1).cpu()
            features.append(feat)
            del batch, _
    handle.remove()
    features = torch.cat(features, dim=0).numpy()
    return features

# ---------- 攻击评估（条件攻击成功率 + Clean/Robust Acc）----------
def evaluate_attack_optimized(attack, name, fwd_model, dataloader, device, num_batches=10):
    """白盒攻击评估，返回 (asr, clean_acc, robust_acc, avg_l2)：
      asr        = 条件攻击成功率 = #(干净分对 且 对抗分错) / #(干净分对)。论文口径：
                   只统计"模型本来分对、被攻击翻掉"的样本，不把本来就分错的算成功。
      clean_acc  = 干净准确率 = #(干净分对) / 总数
      robust_acc = 鲁棒准确率 = #(对抗分对) / 总数
      avg_l2     = 平均（绝对）L2 扰动
      avg_rho    = 平均相对扰动 ρ_adv = mean(‖r‖₂/‖x‖₂)（DeepFool 论文度量）
    注：robust_acc 与 asr 是两套独立指标，robust_acc + asr 不一定等于 100%。
    """
    fwd_model.eval()
    total = 0
    clean_correct = 0      # 干净图分对的数量
    robust_correct = 0     # 对抗图分对的数量
    flipped = 0            # 干净分对里被攻击翻掉的数量
    total_l2 = 0.0
    total_rho = 0.0      # Σ ‖r‖₂/‖x‖₂（DeepFool 论文的相对扰动 ρ_adv）
    # 进度条总长 = 实际评估的 batch 数（min(num_batches, 全部)）；全量评估时 num_batches=len(test_loader)，跑满 10000 张
    eval_batches = min(num_batches, len(dataloader))
    for i, (imgs, lbls) in enumerate(tqdm(dataloader, total=eval_batches, desc=f"Evaluating {name} (≈{eval_batches * dataloader.batch_size} 张)")):
        if i >= num_batches:
            break
        imgs, lbls = imgs.to(device), lbls.to(device)
        adv = attack(imgs, lbls).detach()
        with torch.no_grad():
            clean_ok = (fwd_model(imgs).argmax(dim=1) == lbls)   # 干净图是否分对
            adv_ok = (fwd_model(adv).argmax(dim=1) == lbls)      # 对抗图是否分对
            clean_correct += clean_ok.sum().item()
            robust_correct += adv_ok.sum().item()
            flipped += (clean_ok & ~adv_ok).sum().item()         # 本来对、被翻掉
            total += lbls.size(0)
            total_l2 += torch.norm((adv - imgs).view(imgs.size(0), -1), dim=1).sum().item()
            total_rho += (torch.norm((adv - imgs).view(imgs.size(0), -1), dim=1)
                          / torch.norm(imgs.view(imgs.size(0), -1), dim=1).clamp_min(1e-12)).sum().item()
        del imgs, lbls, adv
    asr = flipped / max(1, clean_correct)
    clean_acc = clean_correct / total
    robust_acc = robust_correct / total
    avg_l2 = total_l2 / total
    avg_rho = total_rho / total          # ρ_adv = 平均相对扰动 ‖r‖₂/‖x‖₂
    return asr, clean_acc, robust_acc, avg_l2, avg_rho

# ---------- 可视化函数 ----------
def tsne_plot_by_class(features, labels, class_names,
                       title="t-SNE of Clean Samples (by class)",
                       save_path=None, perplexity=30, seed=42):
    """对单组特征做 标准化 → PCA → t-SNE，按真实类别上色（颜色用全局 CLASS_COLORS）。"""
    labels = np.asarray(labels).astype(int)
    feats_std = StandardScaler().fit_transform(features)
    n_pca = min(50, feats_std.shape[1], feats_std.shape[0] - 1)
    feats_pca = PCA(n_components=n_pca, random_state=seed).fit_transform(feats_std)
    perp = min(perplexity, max(5, (len(feats_pca) - 1) // 3))
    emb = TSNE(n_components=2, random_state=seed, perplexity=perp,
               init='pca', learning_rate='auto').fit_transform(feats_pca)

    fig, ax = plt.subplots(figsize=(8.5, 7))
    for c in range(len(class_names)):
        m = labels == c
        if not m.any():
            continue
        ax.scatter(emb[m, 0], emb[m, 1], color=CLASS_COLORS[c % 10], s=28, alpha=0.75,
                   edgecolors='white', linewidths=0.4,
                   label=f"{class_names[c]} (n={int(m.sum())})")
    ax.set_title(title, pad=12)
    ax.set_xlabel("t-SNE dimension 1")
    ax.set_ylabel("t-SNE dimension 2")
    ax.tick_params(labelsize=9)
    leg = ax.legend(title="Class", fontsize=9, title_fontsize=11,
                    loc='center left', bbox_to_anchor=(1.01, 0.5))
    leg.get_frame().set_edgecolor('0.8')
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path)
    plt.show()

def tsne_facet_by_class(features_dict, labels_dict, class_names,
                        suptitle="t-SNE by class (shared embedding)",
                        save_path=None, perplexity=30, seed=42,
                        save_each=False):
    """多组特征拟合「同一个」t-SNE 嵌入（坐标可比），再按组分面，面内按真实类别上色。
    颜色统一用全局 CLASS_COLORS（类别索引 → 颜色），保证与其它 t-SNE 图一致。
    features_dict / labels_dict: {组名: ndarray[N, D]} / {组名: ndarray[N]}。
    """
    groups = list(features_dict.keys())
    counts = [features_dict[g].shape[0] for g in groups]
    all_feats = np.concatenate([features_dict[g] for g in groups], axis=0)

    # 全体一起拟合，保证各分面坐标系一致、可直接对比
    feats_std = StandardScaler().fit_transform(all_feats)
    n_pca = min(50, feats_std.shape[1], feats_std.shape[0] - 1)
    feats_pca = PCA(n_components=n_pca, random_state=seed).fit_transform(feats_std)
    perp = min(perplexity, max(5, (len(feats_pca) - 1) // 3))
    emb_all = TSNE(n_components=2, random_state=seed, perplexity=perp,
                   init='pca', learning_rate='auto').fit_transform(feats_pca)

    # 切回每组的二维坐标
    embs, start = {}, 0
    for g, c in zip(groups, counts):
        embs[g] = emb_all[start:start + c]
        start += c

    G = len(groups)
    fig, axes = plt.subplots(1, G, figsize=(4.5 * G, 4.8), sharex=True, sharey=True)
    if G == 1:
        axes = [axes]
    # 统一坐标范围
    xpad = 0.05 * (emb_all[:, 0].max() - emb_all[:, 0].min())
    ypad = 0.05 * (emb_all[:, 1].max() - emb_all[:, 1].min())
    xlim = (emb_all[:, 0].min() - xpad, emb_all[:, 0].max() + xpad)
    ylim = (emb_all[:, 1].min() - ypad, emb_all[:, 1].max() + ypad)

    for ax, g in zip(axes, groups):
        lbl = np.asarray(labels_dict[g]).astype(int)
        e = embs[g]
        for c in range(len(class_names)):
            m = lbl == c
            if m.any():
                ax.scatter(e[m, 0], e[m, 1], color=CLASS_COLORS[c % 10], s=16, alpha=0.75,
                           edgecolors='white', linewidths=0.3)
        ax.set_title(g, fontsize=12)
        ax.set_xlim(xlim)
        ax.set_ylim(ylim)
        ax.set_xlabel("t-SNE dim 1")
        ax.tick_params(labelsize=8)
    axes[0].set_ylabel("t-SNE dim 2")

    # 共享一个类别图例（手工构造，保证 10 类齐全、颜色与散点一致）
    legend_handles = [Line2D([0], [0], marker='o', linestyle='', markersize=7,
                             markerfacecolor=CLASS_COLORS[c % 10], markeredgecolor='white',
                             label=class_names[c]) for c in range(len(class_names))]
    fig.legend(handles=legend_handles, title="Class", loc='center left',
               bbox_to_anchor=(1.0, 0.5), fontsize=9, title_fontsize=11)
    fig.suptitle(suptitle, fontsize=14, fontweight='bold')
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    if save_path:
        os.makedirs(os.path.dirname(save_path) or '.', exist_ok=True)  # 目标子目录不存在时自动创建
        fig.savefig(save_path)

    # save_each=True：把每组（Clean / FGSM ...）各自存成独立矢量图。
    # 仍复用上面「全体一起拟合」的同一套 t-SNE 坐标(embs)与坐标范围(xlim/ylim)，
    # 所以分开的两张图坐标系完全一致、可直接叠放对比。文件名按 save_path 派生：
    # 例如 save_path="a/b/tsne_clean_vs_adv.pdf" → a/b/tsne_clean_vs_adv_Clean.pdf 等。
    if save_each and save_path:
        stem, ext = os.path.splitext(save_path)
        os.makedirs(os.path.dirname(save_path) or '.', exist_ok=True)
        for g in groups:
            f1, a1 = plt.subplots(figsize=(7, 6))
            lbl_g = np.asarray(labels_dict[g]).astype(int)
            e = embs[g]
            for c in range(len(class_names)):
                m = lbl_g == c
                if m.any():
                    a1.scatter(e[m, 0], e[m, 1], color=CLASS_COLORS[c % 10], s=22, alpha=0.78,
                               edgecolors='white', linewidths=0.4, label=class_names[c])
            a1.set_xlim(xlim)
            a1.set_ylim(ylim)
            a1.set_title(f"t-SNE: {g} (by class)", pad=10)
            a1.set_xlabel("t-SNE dim 1")
            a1.set_ylabel("t-SNE dim 2")
            a1.tick_params(labelsize=8)
            leg1 = a1.legend(title="Class", fontsize=8, title_fontsize=10,
                             loc='center left', bbox_to_anchor=(1.01, 0.5))
            leg1.get_frame().set_edgecolor('0.8')
            f1.tight_layout()
            out_path = f"{stem}_{g}{ext}"
            f1.savefig(out_path)
            print(f"  ✓ 单独矢量图已保存：{out_path}")
            plt.close(f1)
    plt.show()

def show_adv_examples(orig_imgs, adv_imgs, labels, attack_name, num=5,
                      model=None, class_names=None, save_path=None):
    """展示对抗前后的图像、扰动，以及模型在 原图 / 对抗图 上的预测。
    标题用 CIFAR-10 类别名（非编号）；预测正确显示绿色、错误/被攻击成功显示红色。
    """
    if model is None:
        model = atk_model
    if class_names is None:
        class_names = CIFAR10_CLASSES
    model.eval()
    n = min(num, len(orig_imgs))

    # 模型对 原图 / 对抗图 的预测（含置信度）
    with torch.no_grad():
        o_prob = torch.softmax(model(orig_imgs[:n].to(device)), dim=1)
        a_prob = torch.softmax(model(adv_imgs[:n].to(device)), dim=1)
    o_pred, o_conf = o_prob.argmax(1).cpu(), o_prob.max(1).values.cpu()
    a_pred, a_conf = a_prob.argmax(1).cpu(), a_prob.max(1).values.cpu()

    fig, axes = plt.subplots(n, 3, figsize=(10.5, 3.5 * n))
    if n == 1:
        axes = axes.reshape(1, -1)
    for i in range(n):
        true_name = class_names[int(labels[i])]
        o_name = class_names[int(o_pred[i])]
        a_name = class_names[int(a_pred[i])]
        orig = np.clip(orig_imgs[i].detach().cpu().numpy().transpose(1, 2, 0), 0, 1)
        adv = np.clip(adv_imgs[i].detach().cpu().numpy().transpose(1, 2, 0), 0, 1)
        diff = adv - orig
        linf = np.abs(diff).max()
        l2 = float(np.linalg.norm(diff))
        pert = np.abs(diff)
        pert = pert / (pert.max() + 1e-12)

        # 原图：真实类别 + 模型预测（正确绿 / 错误红）
        axes[i, 0].imshow(orig)
        axes[i, 0].set_title(f"Original\nTrue: {true_name}\nPred: {o_name} ({o_conf[i]:.0%})",
                             fontsize=10, color=('green' if o_name == true_name else 'red'))
        axes[i, 0].axis('off')

        # 对抗图：攻击后预测（类别被改=攻击成功 标红，否则标绿）
        attacked = (a_name != true_name)
        axes[i, 1].imshow(adv)
        axes[i, 1].set_title(f"{attack_name} Adv\nPred: {a_name} ({a_conf[i]:.0%})",
                             fontsize=10, color=('red' if attacked else 'green'))
        axes[i, 1].axis('off')

        # 扰动图（绝对值归一化显示，标题给出真实 L2 / L∞ 幅度）
        axes[i, 2].imshow(pert, cmap='inferno')
        axes[i, 2].set_title(f"Perturbation\nL2={l2:.2f}, Linf={linf:.3f}", fontsize=10)
        axes[i, 2].axis('off')

    plt.suptitle(f"{attack_name}: Clean vs Adversarial Predictions",
                 fontsize=13, fontweight='bold')
    fig.tight_layout(rect=[0, 0, 1, 0.98])
    if save_path:
        fig.savefig(save_path, dpi=200)
    plt.show()

def plot_l2_hist(l2_dict, title="Perturbation Magnitude Distribution (L2)", save_path=None):
    """多攻击 L2 扰动分布直方图，带均值竖线，统一配色。
    l2_dict: {attack_name: 1D array of L2 norms}
    """
    fig, ax = plt.subplots(figsize=(8, 5))
    for name, l2 in l2_dict.items():
        c = ATTACK_COLORS.get(name)
        l2 = np.asarray(l2)
        ax.hist(l2, bins=30, alpha=0.55, color=c, edgecolor='white', linewidth=0.4,
                label=f"{name} (mean={l2.mean():.3f})")
        ax.axvline(l2.mean(), color=c, linestyle='--', linewidth=1.3, alpha=0.9)
    ax.set_xlabel("L2 norm of perturbation")
    ax.set_ylabel("Frequency")
    ax.set_title(title)
    ax.legend(title="Attack")
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path)
    plt.show()

# ===================== 主流程（优化后） =====================
print("="*60)
print("1. 提取干净样本特征（300个样本）")
clean_feat, clean_label = extract_features(atk_model, test_loader, device, max_samples=300)

print("\n2. 生成 FGSM 对抗样本（1000个样本；t-SNE 仅取前 200 点）")
attack_fgsm = CustomFGSM(atk_model, eps=8 / 255)
fgsm_adv, fgsm_orig, fgsm_lbl, fgsm_l2 = generate_adversarial_optimized(
    attack_fgsm, "FGSM", atk_model, test_loader, device, max_samples=1000)

print("\n3. 提取对抗样本特征")
fgsm_feat = extract_features_from_adv(atk_model, fgsm_adv, batch_size=32)

print("\n4. t-SNE 可视化（按类别上色 + 分面：Clean / FGSM）")
features_dict = {"Clean": clean_feat[:200], "FGSM": fgsm_feat[:200]}
labels_dict   = {"Clean": clean_label[:200], "FGSM": fgsm_lbl[:200]}
tsne_facet_by_class(features_dict, labels_dict, CIFAR10_CLASSES,
                    suptitle="t-SNE by class: Clean vs FGSM (shared embedding)",
                    save_path="tsne_clean_vs_adv.pdf")

print("\n5. 对抗样本可视化")
show_adv_examples(fgsm_orig, fgsm_adv, fgsm_lbl, "FGSM", num=5)

print("\n6. 扰动大小分布")
plot_l2_hist({"FGSM": fgsm_l2}, save_path="l2_distribution.pdf")

print("\n7. 原始模型攻击成功率评估（全量测试集 10000 张）")
# 全量评估：跑遍 test_loader 全部 batch（10000 张）
asr_fgsm, clean_fgsm, rob_fgsm, l2_fgsm, rho_fgsm = evaluate_attack_optimized(attack_fgsm, "FGSM", atk_model, test_loader, device, num_batches=len(test_loader))
print(f"FGSM      : 条件攻击成功率 = {asr_fgsm:.2%}, Clean Acc = {clean_fgsm:.2%}, Robust Acc = {rob_fgsm:.2%}, Avg L2 = {l2_fgsm:.4f}, ρ_adv = {rho_fgsm:.4f}")

print("\n" + "="*60)
print("总结：")
print("- 攻击在 [0,1] 像素空间进行，归一化放进模型，clamp(0,1) 不再破坏图像。")
print("- 提前停止按样本数判断，真正只跑 1000 张，避免整集全跑。")
print("="*60)

In [ ]:
# ==================== 原始（干净）样本的 t-SNE 可视化（按类别上色）====================
# 说明：
#   - 复用 cell-11 提取的 clean_feat / clean_label，以及在 cell-11 定义的
#     tsne_plot_by_class（颜色用全局 CLASS_COLORS，与所有 t-SNE 图保持一致）。
#   - ⚠️ 运行前请先运行 FGSM 单元（cell-11）。
#   - 只看干净样本，按 10 个真实类别上色，观察 ResNet-18 特征空间中各类别的可分性。
print("原始样本 t-SNE（按 CIFAR-10 类别上色）")
tsne_plot_by_class(clean_feat, clean_label, CIFAR10_CLASSES,
                   title="t-SNE of Clean CIFAR-10 Samples (by class)",
                   save_path="tsne_clean_by_class.pdf")

In [ ]:
# ==================== DeepFool 攻击（遵循项目 [0,1] 空间约定）====================
# 说明：
#   - 复用上一单元格定义的 atk_model（内部含归一化）、helper 函数，以及已算出的
#     clean_feat / fgsm_feat / fgsm_l2，避免重复计算。
#   - ⚠️ 运行本单元格前，请先运行 FGSM 单元 与 干净样本 t-SNE 单元（定义 tsne_facet_by_class）。
#   - DeepFool 在 [0,1] 像素空间进行，torchattacks 内部 clamp(0,1) 合法。
import torchattacks

print("="*60)
print("1. 生成 DeepFool 对抗样本（steps=20, overshoot=0.02, 1000个样本；t-SNE 仅取前 200 点）")
# DeepFool 迭代线性化决策边界，寻找最小 L2 扰动；步数降到 20 兼顾速度与显存。
attack_df = torchattacks.DeepFool(atk_model, steps=20, overshoot=0.02)
df_adv, df_orig, df_lbl, df_l2 = generate_adversarial_optimized(
    attack_df, "DeepFool", atk_model, test_loader, device, max_samples=1000)

print("\n2. 提取 DeepFool 对抗样本特征")
df_feat = extract_features_from_adv(atk_model, df_adv, batch_size=32)

print("\n3. t-SNE 可视化（按类别上色 + 分面：Clean / FGSM / DeepFool）")
features_dict = {"Clean": clean_feat[:200], "FGSM": fgsm_feat[:200], "DeepFool": df_feat[:200]}
labels_dict   = {"Clean": clean_label[:200], "FGSM": fgsm_lbl[:200], "DeepFool": df_lbl[:200]}
tsne_facet_by_class(features_dict, labels_dict, CIFAR10_CLASSES,
                    suptitle="t-SNE by class: Clean vs Adversarial (shared embedding)",
                    save_path="tsne_clean_vs_adv.pdf")

print("\n4. DeepFool 对抗样本可视化")
show_adv_examples(df_orig, df_adv, df_lbl, "DeepFool", num=5)

print("\n5. 扰动大小分布（FGSM vs DeepFool）")
plot_l2_hist({"FGSM": fgsm_l2, "DeepFool": df_l2}, save_path="l2_distribution.pdf")

print("\n6. 原始模型攻击成功率评估")
asr_df, clean_df, rob_df, l2_df, rho_df = evaluate_attack_optimized(attack_df, "DeepFool", atk_model, test_loader, device, num_batches=32)
print(f"DeepFool  : 条件攻击成功率 = {asr_df:.2%}, Clean Acc = {clean_df:.2%}, Robust Acc = {rob_df:.2%}, Avg L2 = {l2_df:.4f}, ρ_adv = {rho_df:.4f}")

print("\n" + "="*60)
print("DeepFool 小结：")
print("- DeepFool 通过迭代线性化决策边界寻找最小 L2 扰动，通常扰动远小于 FGSM。")
print("- 攻击在 [0,1] 像素空间进行，归一化放进模型，clamp(0,1) 不破坏图像。")
print("="*60)

In [ ]:
# ==================== APGD 攻击（Auto-PGD，遵循项目 [0,1] 空间约定）====================
# 说明：
#   - 复用前面单元格定义的 atk_model、helper 函数、tsne_facet_by_class，以及已算出的
#     clean_feat / fgsm_* / df_* 结果，做多方对比。
#   - ⚠️ 运行本单元格前，请先运行 FGSM、干净样本 t-SNE、DeepFool 三个单元格。
#   - APGD 是 Linf 预算攻击（自适应步长 + 动量 + 重启），在 [0,1] 像素空间进行，
#     torchattacks 内部 clamp(0,1) 合法。eps=8/255 为 CIFAR 常用的 Linf 预算。
import torchattacks

EPS_LINF = 8 / 255  # Linf 扰动预算

print("="*60)
print(f"1. 生成 APGD 对抗样本（Linf, eps={EPS_LINF:.4f}, steps=20, 1000个样本；t-SNE 仅取前 200 点）")
# APGD 自动调节步长并带重启，是评估鲁棒性的强基线（AutoAttack 的核心组件之一）。
attack_apgd = torchattacks.APGD(atk_model, norm='Linf', eps=EPS_LINF,
                                steps=20, n_restarts=1, loss='ce')
apgd_adv, apgd_orig, apgd_lbl, apgd_l2 = generate_adversarial_optimized(
    attack_apgd, "APGD", atk_model, test_loader, device, max_samples=1000)

print("\n2. 提取 APGD 对抗样本特征")
apgd_feat = extract_features_from_adv(atk_model, apgd_adv, batch_size=32)

print("\n3. t-SNE 可视化（按类别上色 + 分面：Clean / FGSM / DeepFool / APGD）")
features_dict = {"Clean": clean_feat[:200], "FGSM": fgsm_feat[:200], "DeepFool": df_feat[:200], "APGD": apgd_feat[:200]}
labels_dict   = {"Clean": clean_label[:200], "FGSM": fgsm_lbl[:200], "DeepFool": df_lbl[:200], "APGD": apgd_lbl[:200]}
tsne_facet_by_class(features_dict, labels_dict, CIFAR10_CLASSES,
                    suptitle="t-SNE by class: Clean vs FGSM vs DeepFool vs APGD (shared embedding)",
                    save_path="tsne_clean_vs_adv.pdf", save_each=True)

print("\n4. APGD 对抗样本可视化")
show_adv_examples(apgd_orig, apgd_adv, apgd_lbl, "APGD", num=5)

print("\n5. 扰动大小分布（FGSM vs DeepFool vs APGD）")
plot_l2_hist({"FGSM": fgsm_l2, "DeepFool": df_l2, "APGD": apgd_l2}, save_path="l2_distribution.pdf")

print("\n6. 原始模型攻击成功率评估")
asr_apgd, clean_apgd, rob_apgd, l2_apgd, rho_apgd = evaluate_attack_optimized(attack_apgd, "APGD", atk_model, test_loader, device, num_batches=len(test_loader))
print(f"APGD  : 条件攻击成功率 = {asr_apgd:.2%}, Clean Acc = {clean_apgd:.2%}, Robust Acc = {rob_apgd:.2%}, Avg L2 = {l2_apgd:.4f}, ρ_adv = {rho_apgd:.4f}")

print("\n" + "="*60)
print("APGD 小结：")
print("- APGD（Auto-PGD）自适应调节步长并带重启，是比 FGSM 更强的 Linf 攻击。")
print("- 相比 DeepFool（最小 L2 扰动），APGD 在固定 Linf 预算下追求更高攻击成功率。")
print("- 攻击在 [0,1] 像素空间进行，归一化放进模型，clamp(0,1) 不破坏图像。")
print("="*60)

In [ ]:
# ==================== 各攻击方式的混淆矩阵（对抗样本上）====================
# 说明：
#   - 在对抗样本上收集 [真实标签 vs 模型预测]，画混淆矩阵（按 CIFAR-10 类别名）。
#   - 对角线 = 攻击后仍分类正确；非对角 = 被攻击改判成的类别。
#   - 默认按真实类别「行归一化」，更易读每个类别被打到哪些类。
#   - ⚠️ 运行前请先运行已有的攻击单元（FGSM / DeepFool / APGD）。
from sklearn.metrics import confusion_matrix

def plot_attack_confusion(attack, name, fwd_model, dataloader, device,
                          num_batches=10, class_names=None,
                          normalize=True, save_path=None):
    """对抗样本上的混淆矩阵。normalize=True 时按真实类别行归一化。"""
    if class_names is None:
        class_names = CIFAR10_CLASSES
    fwd_model.eval()
    y_true, y_pred = [], []
    eval_batches = min(num_batches, len(dataloader))
    for i, (imgs, lbls) in enumerate(tqdm(dataloader, total=eval_batches, desc=f"Confusion {name}")):
        if i >= num_batches:
            break
        imgs, lbls = imgs.to(device), lbls.to(device)
        adv = attack(imgs, lbls).detach()
        with torch.no_grad():
            pred = fwd_model(adv).argmax(1)
        y_true.append(lbls.cpu())
        y_pred.append(pred.cpu())
        del imgs, lbls, adv
    y_true = torch.cat(y_true).numpy()
    y_pred = torch.cat(y_pred).numpy()

    K = len(class_names)
    cm = confusion_matrix(y_true, y_pred, labels=list(range(K)))
    if normalize:
        row = cm.sum(axis=1, keepdims=True)
        cmn = np.divide(cm, row, out=np.zeros(cm.shape, dtype=float), where=row != 0)
        vmax = 1.0
    else:
        cmn = cm.astype(float)
        vmax = cm.max()

    fig, ax = plt.subplots(figsize=(7.5, 6.5))
    im = ax.imshow(cmn, cmap='Blues', vmin=0, vmax=vmax)
    ax.set_xticks(range(K))
    ax.set_yticks(range(K))
    ax.set_xticklabels(class_names, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(class_names, fontsize=9)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    acc = float((y_true == y_pred).mean())
    ax.set_title(f"{name}: Confusion Matrix on Adversarial Samples\n"
                 f"(adv accuracy = {acc:.1%}, adv error = {1 - acc:.1%})")
    thresh = vmax / 2
    for r in range(K):
        for c in range(K):
            v = cmn[r, c]
            if v > 0:
                txt = f"{v:.2f}" if normalize else f"{int(cm[r, c])}"
                ax.text(c, r, txt, ha='center', va='center', fontsize=7,
                        color='white' if v > thresh else 'black')
    ax.grid(False)  # 全局网格会盖在热力图上，这里关掉
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04,
                 label='Row-normalized rate' if normalize else 'Count')
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path)
    plt.show()

# 自动收集已经运行过的攻击对象（缺哪个就跳过哪个）
_attacks = []
for _var, _name in [('attack_fgsm', 'FGSM'), ('attack_df', 'DeepFool'), ('attack_apgd', 'APGD')]:
    if _var in globals():
        _attacks.append((globals()[_var], _name))

if not _attacks:
    print("⚠️ 未找到任何攻击对象，请先运行 FGSM / DeepFool / APGD 单元。")
for _atk, _name in _attacks:
    print(f"\n>>> {_name} 混淆矩阵（对抗样本）")
    plot_attack_confusion(_atk, _name, atk_model, test_loader, device,
                          num_batches=10, save_path=f"confusion_{_name.lower()}.pdf")

In [ ]:
# ==================== PGD 对抗训练（全量微调，PGD-7）====================
# 说明：
#   - 防御思路：用 PGD 现场生成的对抗样本微调模型，逼它「被攻击也分对」，得到鲁棒模型。
#   - 基于「优化版本」单元已加载的干净训练 model 克隆一份做对抗微调，原 model / atk_model
#     保持不变，方便最后对比「普通模型 vs 鲁棒模型」。
#   - 攻击与训练全部在 [0,1] 像素空间进行（与项目约定一致），归一化在 atk_model 内部完成。
#   - 【已训练好】下方 SKIP_AT_TRAINING=True 时跳过 80 轮对抗训练，直接加载已有鲁棒权重；
#     与 cell 4 跳过基础训练的做法一致。要重新训练就把它设为 False。
#   - 防塌缩配方：干净/对抗各半损失 + eps 预热 + 梯度裁剪 + 每轮记录 clean/adv acc。
#   - ⚠️ 运行前请先运行「优化版本」单元（定义 model / device / Normalize / NORM_MEAN /
#     NORM_STD / DATA_ROOT / CustomFGSM / evaluate_attack_optimized / test_loader）。
import os
import copy
import random
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import DataLoader, Subset
from torchvision import transforms, datasets
from tqdm import tqdm

# ---------- 超参数 ----------
TRAIN_SUBSET  = None      # None = 用全量训练集（50000）；也可填整数只取子集
EPOCHS        = 80        # 对抗训练收敛慢：标准 PGD-AT 训 80 轮
BATCH_SIZE    = 128
EPS           = 8 / 255   # Linf 扰动预算（与 APGD 评估一致）
PGD_STEPS     = 7         # 内层 PGD 迭代步数（PGD-7）
WARMUP_EPOCHS = 5         # eps 预热轮数
LR            = 0.1       # 标准 PGD-AT 大 LR=0.1 + 多步衰减
MILESTONES    = [40, 60]  # 在 50%(40) / 75%(60) 处把 LR ÷10
LR_GAMMA      = 0.1
LAMBDA_ADV    = 0.5       # 损失 = (1-λ)·CE(clean) + λ·CE(adv)，干净样本兜底防塌缩
CLIP_NORM     = 1.0       # 梯度裁剪上限
SEED          = 42
ROBUST_CKPT   = "resnet18_cifar10_pgd_at.pth"   # 训练时鲁棒模型保存路径
# 已有训练好的鲁棒模型 → 跳过对抗训练，直接加载（与 cell 4 跳过基础训练一致）
SKIP_AT_TRAINING = True
ROBUST_CKPT_LOAD = "/kaggle/input/notebooks/liangliguo/cifar/resnet18_cifar10_pgd_at.pth"

torch.manual_seed(SEED)
random.seed(SEED)

# ---------- 克隆预训练模型做鲁棒微调（原 model / atk_model 保持不变作基线）----------
robust_net = copy.deepcopy(model).to(device)
atk_model_robust = nn.Sequential(Normalize(NORM_MEAN, NORM_STD), robust_net).to(device)

# ---------- PGD 攻击（Linf, [0,1] 空间）----------
def pgd_attack_linf(fwd_model, imgs, labels, eps, alpha, steps=PGD_STEPS):
    was_training = fwd_model.training
    fwd_model.eval()
    delta = torch.empty_like(imgs).uniform_(-eps, eps)
    x_adv = torch.clamp(imgs + delta, 0, 1).detach()
    for _ in range(steps):
        x_adv.requires_grad_(True)
        loss = nn.functional.cross_entropy(fwd_model(x_adv), labels)
        grad = torch.autograd.grad(loss, x_adv)[0]
        x_adv = x_adv.detach() + alpha * grad.sign()
        x_adv = torch.min(torch.max(x_adv, imgs - eps), imgs + eps)  # 投影回 Linf 球
        x_adv = torch.clamp(x_adv, 0, 1).detach()                    # 裁剪到合法图像
    if was_training:
        fwd_model.train()
    return x_adv

history = {"epoch": [], "loss": [], "clean_acc": [], "adv_acc": [], "lr": []}

if SKIP_AT_TRAINING and os.path.exists(ROBUST_CKPT_LOAD):
    # ===== 跳过训练：直接加载已有鲁棒模型 =====
    print(f"⏭️  跳过对抗训练，直接加载已有鲁棒模型: {ROBUST_CKPT_LOAD}")
    ckpt = torch.load(ROBUST_CKPT_LOAD, map_location=device)
    robust_net.load_state_dict(ckpt['model_state_dict'])
    robust_net.eval()
    print("✓ 鲁棒模型加载完成")
else:
    # ===== 从头做 PGD 对抗训练（SKIP_AT_TRAINING=False 或找不到权重时）=====
    # ---------- 训练数据：CIFAR-10 训练集（保持 [0,1]，加 CIFAR 增广）----------
    train_transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),   # → [0,1]，归一化交给 atk_model 内部完成
    ])
    _need_dl = not os.path.isdir(os.path.join(DATA_ROOT, 'cifar-10-batches-py'))
    train_dataset = datasets.CIFAR10(root=DATA_ROOT, train=True,
                                     download=_need_dl, transform=train_transform)
    if TRAIN_SUBSET is None:
        train_data = train_dataset                       # 全量 50000
    else:
        _g = torch.Generator().manual_seed(SEED)
        subset_idx = torch.randperm(len(train_dataset), generator=_g)[:TRAIN_SUBSET].tolist()
        train_data = Subset(train_dataset, subset_idx)   # 仅取子集
    train_loader = DataLoader(train_data,
                              batch_size=BATCH_SIZE, shuffle=True, num_workers=4,
                              pin_memory=True, persistent_workers=True, drop_last=True)

    print(f"ℹ 单卡训练，device={device}，batch={BATCH_SIZE}")

    optimizer = torch.optim.SGD(robust_net.parameters(), lr=LR, momentum=0.9, weight_decay=5e-4)
    scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=MILESTONES, gamma=LR_GAMMA)
    criterion = nn.CrossEntropyLoss()

    print("=" * 60)
    print(f"PGD 对抗训练：样本数={len(train_data)}, epochs={EPOCHS}, PGD-{PGD_STEPS}, "
          f"eps={EPS:.4f}, LR={LR}, milestones={MILESTONES}, lambda_adv={LAMBDA_ADV}")
    for epoch in range(1, EPOCHS + 1):
        # eps 预热：前 WARMUP_EPOCHS 轮从 0.5×eps 线性升到 1×eps，步长随 eps 缩放
        cur_eps = EPS * (0.5 + 0.5 * min(1.0, (epoch - 1) / max(1, WARMUP_EPOCHS - 1)))
        cur_alpha = 2.5 * cur_eps / PGD_STEPS

        robust_net.train()
        run_loss, clean_correct, adv_correct, seen = 0.0, 0, 0, 0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} (eps={cur_eps:.4f})")
        for imgs, lbls in pbar:
            imgs = imgs.to(device, non_blocking=True)
            lbls = lbls.to(device, non_blocking=True)
            # 1) 用当前模型现场把这批干净图打成对抗样本
            x_adv = pgd_attack_linf(atk_model_robust, imgs, lbls, eps=cur_eps, alpha=cur_alpha)
            # 2) 干净 + 对抗 混合损失（标签都是原始 y）
            robust_net.train()
            optimizer.zero_grad()
            out_clean = atk_model_robust(imgs)
            out_adv = atk_model_robust(x_adv)
            loss = (1 - LAMBDA_ADV) * criterion(out_clean, lbls) + LAMBDA_ADV * criterion(out_adv, lbls)
            loss.backward()
            clip_grad_norm_(robust_net.parameters(), CLIP_NORM)
            optimizer.step()
            # 统计
            run_loss += loss.item() * lbls.size(0)
            clean_correct += (out_clean.argmax(1) == lbls).sum().item()
            adv_correct += (out_adv.argmax(1) == lbls).sum().item()
            seen += lbls.size(0)
            pbar.set_postfix(loss=f"{run_loss/seen:.3f}",
                             clean=f"{clean_correct/seen:.2%}",
                             adv=f"{adv_correct/seen:.2%}")
            del imgs, lbls, x_adv, out_clean, out_adv, loss
        cur_lr = scheduler.get_last_lr()[0]
        print(f"  Epoch {epoch}: loss={run_loss/seen:.4f}, "
              f"clean-acc={clean_correct/seen:.2%}, adv-acc={adv_correct/seen:.2%}, "
              f"lr={cur_lr:.5f}")
        history["epoch"].append(epoch)
        history["loss"].append(run_loss / seen)
        history["clean_acc"].append(clean_correct / seen)
        history["adv_acc"].append(adv_correct / seen)
        history["lr"].append(cur_lr)
        scheduler.step()

    robust_net.eval()
    torch.save({'model_state_dict': robust_net.state_dict()}, ROBUST_CKPT)
    print(f"\n✓ 鲁棒模型已保存：{ROBUST_CKPT}")

    # ---------- 绘制训练曲线（clean-acc / adv-acc / loss）→ 矢量图 ----------
    ep = history["epoch"]
    fig, ax1 = plt.subplots(figsize=(8.5, 5))
    l1, = ax1.plot(ep, [a * 100 for a in history["clean_acc"]], color="#4C72B0",
                   marker="o", ms=3, lw=1.8, label="Train clean-acc")
    l2, = ax1.plot(ep, [a * 100 for a in history["adv_acc"]], color="#C44E52",
                   marker="s", ms=3, lw=1.8, label="Train adv-acc (PGD-7)")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Accuracy (%)")
    ax1.set_ylim(0, 100)
    ax2 = ax1.twinx()
    l3, = ax2.plot(ep, history["loss"], color="#8C8C8C", lw=1.4, ls="--", label="Train loss")
    ax2.set_ylabel("Loss")
    ax2.grid(False)
    for ms in MILESTONES:
        if ms <= EPOCHS:
            ax1.axvline(ms, color="0.6", ls=":", lw=1)
    ax1.set_title("PGD-AT Training Curves: clean vs adversarial accuracy", pad=12)
    ax1.legend(handles=[l1, l2, l3], loc="lower right")
    fig.tight_layout()
    fig.savefig("pgd_at_training_curves.svg")
    fig.savefig("pgd_at_training_curves.pdf")
    print("✓ 训练曲线矢量图已保存：pgd_at_training_curves.svg / .pdf")
    plt.show()

# ---------- 评估：Clean Acc + Robust Acc + 条件攻击成功率（普通 vs 鲁棒，白盒 FGSM）----------
print("\n" + "=" * 60)
print("评估（FGSM, eps=8/255, 白盒）：")
print(f"{'模型':<8}{'Clean Acc':>12}{'Robust Acc':>12}{'攻击成功率':>12}")
for tag, fm in [("普通模型", atk_model), ("鲁棒模型", atk_model_robust)]:
    atk = CustomFGSM(fm, eps=8 / 255)
    asr, clean_acc, rob_acc, _, _ = evaluate_attack_optimized(atk, "FGSM", fm, test_loader, device, num_batches=len(test_loader))  # 全量 10000
    print(f"{tag:<8}{clean_acc:>11.2%}{rob_acc:>12.2%}{asr:>12.2%}")
print("提示：Clean 和 Robust 都高才是真鲁棒；两者都低 = 模型塌缩。")
print("攻击成功率为条件口径（只统计干净分对里被翻掉的），故 Robust Acc + 攻击成功率 不一定 = 100%。")
print("完整强弱攻击对比（FGSM / PGD-20 / APGD）见下一单元。")
print("=" * 60)


In [ ]:
# ==================== 诚实评估：普通模型 vs 鲁棒模型（FGSM + PGD-20 + APGD）====================
# 说明：
#   - 【改进2：加一档 PGD-20】攻击由弱到强排成阶梯：FGSM(单步,弱) < PGD-20(20步,标准基准)
#     < APGD(自适应,最强)。PGD-20 是学术界事实标准，便于和论文结果对齐；三者 robust acc
#     应「依次递减」，若不递减说明评估有 bug（如梯度混淆）。
#   - 复用前面定义的 atk_model（普通）/ atk_model_robust（鲁棒）/ CustomFGSM /
#     evaluate_attack_optimized / test_loader。
#   - 【输出矢量图】把三种攻击下两模型的 Robust Acc 画成分组柱状图，存 SVG/PDF。
#   - ⚠️ 运行前请先运行「优化版本」单元 和「PGD 对抗训练」单元。
import numpy as np
import matplotlib.pyplot as plt
import torchattacks

# 每种攻击都按"白盒"对当前被评估模型单独构造（attack 与 fwd_model 一致）
def _build_fgsm(fm):
    return CustomFGSM(fm, eps=8 / 255)                                  # 单步弱攻击

def _build_pgd20(fm):
    # 改进2：标准 PGD-20（Linf, eps=8/255, step=2/255, 随机起点），学术界通用基准
    return torchattacks.PGD(fm, eps=8/255, alpha=2/255, steps=20, random_start=True)

def _build_apgd(fm):
    # Auto-PGD：AutoAttack 的核心组件，自适应步长+重启，远强于 FGSM
    return torchattacks.APGD(fm, norm='Linf', eps=8/255, steps=20, n_restarts=1, loss='ce')

ATTACKS = [("FGSM  (单步, 弱)", _build_fgsm, "FGSM"),
           ("PGD-20 (Linf 8/255, 标准)", _build_pgd20, "PGD-20"),
           ("APGD  (Linf 8/255, 强)", _build_apgd, "APGD")]
MODELS = [("普通模型", atk_model), ("鲁棒模型", atk_model_robust)]

# 评估样本数 = EVAL_BATCHES × batch_size(32)。这里用全量测试集（test_loader 现 10000 张），
# 是鲁棒性评估金标准；DeepFool 因太慢仍保留 1000（见其单元）。
EVAL_BATCHES = len(test_loader)

# results[short_name][模型tag] = (clean_acc, robust_acc, asr)
results = {short: {} for _, _, short in ATTACKS}

print("=" * 66)
print("普通模型 vs 鲁棒模型：Clean / Robust Acc 对比（FGSM < PGD-20 < APGD）")
for atk_name, builder, short in ATTACKS:
    print("-" * 66)
    print(f"[{atk_name}]")
    print(f"{'模型':<8}{'Clean Acc':>12}{'Robust Acc':>12}{'攻击成功率':>12}")
    for tag, fm in MODELS:
        atk = builder(fm)
        asr, clean_acc, rob_acc, _, _ = evaluate_attack_optimized(
            atk, atk_name, fm, test_loader, device, num_batches=EVAL_BATCHES)
        print(f"{tag:<8}{clean_acc:>11.2%}{rob_acc:>12.2%}{asr:>12.2%}")
        results[short][tag] = (clean_acc, rob_acc, asr)
print("-" * 66)
print("解读：")
print("- 强度阶梯 FGSM < PGD-20 < APGD：同一模型的 robust acc 应依次下降，否则评估存疑。")
print("- 对抗训练若真有效：鲁棒模型在三种攻击下的 robust acc 都应高于普通模型。")
print("- PGD-20 是论文标准基准，可直接和文献中的 robust acc 数值对比。")
print("- 攻击成功率为条件口径，故 Robust Acc + 攻击成功率 不一定 = 100%。")
print("=" * 66)

# ---------- 输出矢量图：三种攻击下两模型的 Robust Acc 分组柱状图 ----------
attack_order = [short for _, _, short in ATTACKS]
x = np.arange(len(attack_order))
width = 0.36
plain_rob = [results[s]["普通模型"][1] * 100 for s in attack_order]
robust_rob = [results[s]["鲁棒模型"][1] * 100 for s in attack_order]
plain_clean = results[attack_order[0]]["普通模型"][0] * 100   # clean 与攻击无关，取其一
robust_clean = results[attack_order[0]]["鲁棒模型"][0] * 100

fig, ax = plt.subplots(figsize=(8.5, 5))
b1 = ax.bar(x - width / 2, plain_rob, width, label="Standard model", color="#8C8C8C", edgecolor="white")
b2 = ax.bar(x + width / 2, robust_rob, width, label="Robust model (PGD-AT)", color="#C44E52", edgecolor="white")
# 干净准确率参考线（两模型各一条），标出鲁棒性提升的「天花板」（图内用英文，避免缺中文字体显示方块）
ax.axhline(plain_clean, color="#8C8C8C", ls="--", lw=1, alpha=0.8)
ax.axhline(robust_clean, color="#C44E52", ls="--", lw=1, alpha=0.8)
ax.text(len(attack_order) - 0.5, plain_clean + 1, f"Standard clean {plain_clean:.0f}%",
        color="#5A5A5A", fontsize=8, ha="right")
ax.text(len(attack_order) - 0.5, robust_clean + 1, f"Robust clean {robust_clean:.0f}%",
        color="#C44E52", fontsize=8, ha="right")
for bars in (b1, b2):
    for r in bars:
        ax.annotate(f"{r.get_height():.1f}", (r.get_x() + r.get_width() / 2, r.get_height()),
                    textcoords="offset points", xytext=(0, 3), ha="center", fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(attack_order)
ax.set_ylabel("Robust Accuracy (%)")
ax.set_ylim(0, 100)
ax.set_title("Robust Accuracy vs Attack Strength (FGSM < PGD-20 < APGD)", pad=12)
ax.legend(loc="upper right")
fig.tight_layout()
fig.savefig("robustness_comparison.svg")
fig.savefig("robustness_comparison.pdf")
print("✓ 鲁棒性对比矢量图已保存：robustness_comparison.svg / .pdf")
plt.show()


In [ ]:
# ==================== DeepFool 最小扰动：普通模型 vs 鲁棒模型 ====================
# 口径说明（与上一单元的 Linf 阶梯互补）：
#   - DeepFool 求「刚好越过决策边界」的最小扰动，没有固定预算，因此对鲁棒性的
#     正确度量是「平均 / 中位扰动大小」——越大说明样本离决策边界越远、模型越鲁棒。
#   - 同时报两种口径：绝对 L2 = ‖r‖₂，相对扰动 ρ_adv = ‖r‖₂/‖x‖₂（DeepFool 论文度量，
#     与「鲁棒性度量」页公式一致）。CIFAR 上 ‖x‖≈常数，二者同向，比值不变。
#   - 不能用 robust acc 比：DeepFool 几乎总能越界（成功率近 100%），robust acc 不可比。
#   - ⚠️ 运行前需已有 atk_model（普通）、atk_model_robust（鲁棒）、test_loader 及前面定义的 helper。
import torch
import numpy as np
import torchattacks

def deepfool_pert_stats(fwd_model, name, max_samples=1000):
    """跑 DeepFool，在「干净分对且被翻掉」样本上返回 (L2数组, ρ_adv数组, 翻转率, Clean Acc)。"""
    atk = torchattacks.DeepFool(fwd_model, steps=20, overshoot=0.02)
    adv, orig, lbl, l2 = generate_adversarial_optimized(
        atk, name, fwd_model, test_loader, device, max_samples=max_samples)
    fwd_model.eval()
    with torch.no_grad():
        clean_ok = (fwd_model(orig.to(device)).argmax(1).cpu() == lbl)
        adv_pred = fwd_model(adv.to(device)).argmax(1).cpu()
    flipped = (clean_ok & (adv_pred != lbl)).numpy()      # 最小扰动口径：仅统计本来分对、被翻掉的
    x_norm = torch.norm(orig.reshape(orig.size(0), -1), dim=1).numpy()
    rho = l2 / np.clip(x_norm, 1e-12, None)               # ρ_adv = ‖r‖₂/‖x‖₂
    return l2[flipped], rho[flipped], flipped.mean(), clean_ok.float().mean().item()

print("=" * 80)
print("DeepFool 最小扰动对比：普通模型 vs 鲁棒模型（前 1000 样本，扰动越大越鲁棒）")
print("-" * 80)
print(f"{'模型':<8}{'Clean Acc':>11}{'翻转率':>9}{'平均 L2':>10}{'中位 L2':>10}{'平均 ρ_adv':>13}{'中位 ρ_adv':>13}")
df_rho_models = {}
for cn_name, en_name, m in [("普通模型", "Base", atk_model),
                            ("鲁棒模型", "Robust", atk_model_robust)]:
    l2_f, rho_f, flip_rate, clean_acc = deepfool_pert_stats(m, f"DeepFool/{en_name}")
    df_rho_models[en_name] = rho_f
    print(f"{cn_name:<8}{clean_acc:>10.2%}{flip_rate:>9.2%}"
          f"{l2_f.mean():>10.4f}{np.median(l2_f):>10.4f}"
          f"{rho_f.mean():>13.5f}{np.median(rho_f):>13.5f}")
print("-" * 80)
print("解读：鲁棒模型的平均/中位扰动（L2 与 ρ_adv 同向）应明显大于普通模型——把它骗倒需要更大扰动，")
print("      即对抗训练把决策边界从样本旁推开；与 Linf 口径下 robust acc 的提升相互印证。")
print("=" * 80)

# 直方图对比（仅成功翻转样本的 ρ_adv 分布；英文标签避免中文缺字）
fig, ax = plt.subplots(figsize=(8, 5))
for en_name, color in [("Base", "#4C72B0"), ("Robust", "#C44E52")]:
    v = df_rho_models[en_name]
    ax.hist(v, bins=30, alpha=0.55, color=color, edgecolor='white', linewidth=0.4,
            label=f"{en_name} (mean={v.mean():.5f}, median={np.median(v):.5f})")
    ax.axvline(v.mean(), color=color, linestyle='--', linewidth=1.3, alpha=0.9)
ax.set_xlabel(r"DeepFool relative perturbation  $\rho_{adv}=\|r\|_2/\|x\|_2$")
ax.set_ylabel("Frequency")
ax.set_title("DeepFool minimum relative perturbation: base vs robust model")
ax.legend(title="Model")
fig.tight_layout()
fig.savefig("deepfool_rho_base_vs_robust.pdf")
print("✓ 矢量图已保存：deepfool_rho_base_vs_robust.pdf")
plt.show()
